# Quickstart DataFrame

## Create SparkSession

In [ ]:
from pyspark.sql import SparkSession, Row, Column
import os
from datetime import datetime, date
import pandas as pd
from pyspark.sql.functions import upper, pandas_udf

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=date(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=date(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=date(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
])

df

In [ ]:
# All DataFrames above result same.
df.show()
df.printSchema()

In [ ]:
pandas_df = pd.DataFrame({
    'a': [1, 2, 3],
    'b': [2., 3., 4.],
    'c': ['string1', 'string2', 'string3'],
    'd': [date(2000, 1, 1), date(2000, 2, 1), date(2000, 3, 1)],
    'e': [datetime(2000, 1, 1, 12, 0), datetime(2000, 1, 2, 12, 0), datetime(2000, 1, 3, 12, 0)]
})
df = spark.createDataFrame(pandas_df)
df.show()
df.printSchema()

Alternatively, you can enable spark.sql.repl.eagerEval.enabled configuration for the eager evaluation of PySpark DataFrame in notebooks such as Jupyter. The number of rows to show can be controlled via spark.sql.repl.eagerEval.maxNumRows configuration.

In [ ]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [ ]:
df

In [ ]:
df.select("a", "b", "c").describe().show()

## Récuperation des données.

In [ ]:
df.collect() # DataFrame.collect() returns a list of Row objects (can be thrown out of memory if too large)

In [ ]:
df.take(2) # DataFrame.take(n) returns n Row objects (help to avoid thrown out of memory)

In [ ]:
df.head(2) # DataFrame.head(n) returns first n Row objects

In [ ]:
df.tail(2) # DataFrame.tail(n) returns last n Row objects

## Columns 

In [ ]:
type(df.c) == type(upper(df.c)) == type(df.c.isNull())

In [ ]:
df.select(df.c).show()

In [ ]:
# Assign new Column instance.
df.withColumn('upper_c', upper(df.c)).show()

In [ ]:
# To select a subset of rows, use DataFrame.filter().
df.filter(df.a > 1).show()

In [ ]:
df.filter(df.a == 1).show()

## Applying function

In [ ]:
@pandas_udf('long')
def pandas_plus_one(series: pd.Series) -> pd.Series:
    # Simply plus one by using pandas Series.
    return series + 1

df.select(pandas_plus_one(df.a)).show()

In [ ]:
def pandas_filter_func(iterator):
    for pandas_df in iterator:
        yield pandas_df[pandas_df.a == 1]

df.mapInPandas(pandas_filter_func, schema=df.schema).show()

## Grouping

In [ ]:
df = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])
df.show()

In [ ]:
df.groupBy('color').avg().show()

In [ ]:
df.groupBy('color').avg('v1', 'v2').show()

In [ ]:
df.groupBy('color').agg({'v1': 'avg', 'v2': 'sum'}).show()

In [ ]:
def plus_mean(pandas_df):
    return pandas_df.assign(v1=pandas_df.v1 - pandas_df.v1.mean())

df.groupby('color').applyInPandas(plus_mean, schema=df.schema).show()

In [ ]:
df1 = spark.createDataFrame(
    [(20000101, 1, 1.0), (20000101, 2, 2.0), (20000102, 1, 3.0), (20000102, 2, 4.0)],
    ('time', 'id', 'v1'))

df2 = spark.createDataFrame(
    [(20000101, 1, 'x'), (20000101, 2, 'y')],
    ('time', 'id', 'v2'))

def merge_ordered(l, r):
    return pd.merge_ordered(l, r)

df1.groupby('id').cogroup(df2.groupby('id')).applyInPandas(merge_ordered, schema='time int, id int, v1 double, v2 string').show()